# Bundle Migration: Generate & Deploy

## Overview

Deploy dashboards to target workspace with **dual deployment options**:
- **SDK Direct**: All resources via API (dashboards, permissions, schedules) - Recommended
- **Asset Bundle + SDK**: Dashboards via bundle, schedules via SDK - For IaC workflows

## Prerequisites

- ✅ Bundle_03 completed (dashboards exported and transformed)
- ✅ databricks.yml configured with target workspace details
- ✅ Secrets configured for cross-workspace authentication
- ✅ See PREREQUISITES_AND_SETUP.md for complete setup

## Features

- ✅ Supports both CLI and Interactive execution
- ✅ Dry run mode for safe testing
- ✅ Unified deployment packages (dashboards + permissions + schedules)
- ✅ Fallback authentication for cross-workspace access
- ✅ Comprehensive verification and reporting

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml databricks-cli "numpy<2" --quiet
dbutils.library.restartPython()

## Cell 0.5: Configuration Display

**All configuration comes from databricks.yml (single source of truth)**

This notebook does NOT have hardcoded defaults. All parameters are passed via `base_parameters` from `databricks.yml`.

**CLI Usage:**
```bash
# Dry run (safe default from YAML)
databricks bundle run generate_deploy -t dev --profile source-workspace

# Live deploy (override dry_run_mode only)
databricks bundle run generate_deploy -t dev --var="dry_run_mode=false" --profile source-workspace
```

**To change deployment method, auth method, or workspace configs:** Edit `databricks.yml` and redeploy bundle.

---

In [ ]:
# ============================================================================
# DISPLAY CONFIGURATION (Read-Only)
# ============================================================================
# This cell simply displays the configuration that was passed via base_parameters
# from databricks.yml. NO defaults are set here - everything comes from YAML.
#
# When run via CLI: base_parameters from databricks.yml are injected
# When run interactively: User must ensure notebook has parameters set
# ============================================================================

print("=" * 80)
print("CONFIGURATION LOADED FROM DATABRICKS.YML")
print("=" * 80)
print()
print("✓ All configuration comes from databricks.yml (single source of truth)")
print("✓ Only dry_run_mode can be overridden via CLI --var flag")
print("✓ No hardcoded defaults in notebook")
print()
print("=" * 80)

## Cell 1: Import Helpers & Setup

In [ ]:
import sys
import os
from pathlib import Path

print("=" * 80)
print("SETTING UP HELPERS PATH")
print("=" * 80)

try:
    # Get notebook path in Databricks
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    
    # Add /Workspace prefix if not present
    if not notebook_path.startswith('/Workspace'):
        notebook_path = '/Workspace' + notebook_path
    
    # Get parent directory of Bundle folder
    notebook_dir = os.path.dirname(notebook_path)
    bundle_parent = os.path.dirname(notebook_dir)
    
    # Add to sys.path
    if bundle_parent not in sys.path:
        sys.path.insert(0, bundle_parent)
    
    print(f"✅ Helpers path: {bundle_parent}/helpers")
    
except Exception as e:
    print(f"⚠️  Could not auto-detect path: {e}")
    print("   Attempting fallback...")
    # Fallback for local testing
    sys.path.insert(0, os.path.abspath('../'))

# Import all helpers
from helpers import (
    set_config,
    get_config,
    ensure_directory_exists,
    create_target_workspace_client,
    build_deployment_packages,
    deploy_via_sdk,
    resolve_warehouse,
    generate_bundle_structure,
    validate_bundle,
    deploy_bundle
)

import pandas as pd
import json

print("\n✅ All imports successful")
print("=" * 80)

## Cell 2: Load Configuration

This cell reads all parameters from widgets (set by CLI or interactive UI) and builds the config dictionary.

In [ ]:
print("=" * 80)
print("LOADING CONFIGURATION FROM PARAMETERS")
print("=" * 80)

# Read all parameters from widgets (passed from databricks.yml or set interactively)
catalog = dbutils.widgets.get('catalog')
volume_base = dbutils.widgets.get('volume_base')
source_workspace_url = dbutils.widgets.get('source_workspace_url')
target_workspace_url = dbutils.widgets.get('target_workspace_url')
exported_path_rel = dbutils.widgets.get('exported_path')
transformed_path_rel = dbutils.widgets.get('transformed_path')
bundles_path_rel = dbutils.widgets.get('bundles_path')
target_parent_path = dbutils.widgets.get('target_parent_path')
warehouse_id = dbutils.widgets.get('warehouse_id')  # Preferred: Direct warehouse ID
warehouse_name = dbutils.widgets.get('warehouse_name')  # Fallback: Warehouse name
bundle_name = dbutils.widgets.get('bundle_name')

# Options
apply_permissions = dbutils.widgets.get('apply_permissions').lower() == 'true'
permissions_dry_run = dbutils.widgets.get('permissions_dry_run').lower() == 'true'
apply_schedules = dbutils.widgets.get('apply_schedules').lower() == 'true'
schedules_dry_run = dbutils.widgets.get('schedules_dry_run').lower() == 'true'
embed_credentials = dbutils.widgets.get('embed_credentials').lower() == 'true'

# NEW: Deployment method parameters
deployment_method = dbutils.widgets.get('deployment_method')  # 'sdk_direct' or 'asset_bundle'
dry_run_mode = dbutils.widgets.get('dry_run_mode').lower() == 'true'
target_workspace_secret_scope = dbutils.widgets.get('target_workspace_secret_scope')

# Cross-workspace authentication parameters
try:
    auth_method = dbutils.widgets.get('auth_method')  # 'pat' or 'sp_oauth'
except:
    auth_method = 'pat'  # Default to PAT for backward compatibility

try:
    sp_secret_scope = dbutils.widgets.get('sp_secret_scope')
except:
    sp_secret_scope = target_workspace_secret_scope  # Fallback to existing scope

# Performance options
try:
    skip_duplicate_check = dbutils.widgets.get('skip_duplicate_check').lower() == 'true'
except:
    skip_duplicate_check = False  # Default: check for duplicates

# Build full paths
exported_path = f"{volume_base}/{exported_path_rel}"
transformed_path = f"{volume_base}/{transformed_path_rel}"
bundles_path = f"{volume_base}/{bundles_path_rel}"

# Build config dict for helper functions
config = {
    'source': {
        'workspace_url': source_workspace_url,
        'auth': {'method': 'oauth'}
    },
    'target': {
        'workspace_url': target_workspace_url,
        'auth': {'method': 'oauth'}
    },
    'paths': {
        'volume_base': volume_base,
        'exported': exported_path_rel,
        'transformed': transformed_path_rel,
        'bundles': bundles_path_rel,
        'target_parent_path': target_parent_path
    },
    'bundle': {
        'name': bundle_name,
        'embed_credentials': embed_credentials
    },
    'warehouse': {
        'warehouse_id': warehouse_id,
        'warehouse_name': warehouse_name
    },
    'permissions': {
        'apply': apply_permissions,
        'dry_run': permissions_dry_run
    },
    'schedules': {
        'apply': apply_schedules,
        'dry_run': schedules_dry_run
    },
    'deployment': {
        'method': deployment_method,
        'dry_run': dry_run_mode,
        'secret_scope': target_workspace_secret_scope
    },
    'auth': {
        'method': auth_method,
        'sp_secret_scope': sp_secret_scope
    }
}

# Cache config for helper functions
set_config(config)

print("\n✅ Configuration loaded from parameters\n")
print(f"   Source: {source_workspace_url}")
print(f"   Target: {target_workspace_url}")
print(f"   Volume base: {volume_base}")
print(f"   Bundle name: {bundle_name}")
print(f"   Target path: {target_parent_path}")
print(f"   Warehouse: {warehouse_id if warehouse_id else warehouse_name} {'(ID)' if warehouse_id else '(name)'}")
print(f"   Deployment method: {deployment_method.upper()}")
print(f"   Auth method: {auth_method.upper()} ({'Service Principal OAuth M2M' if auth_method == 'sp_oauth' else 'PAT Token'})")
print(f"   Dry run mode: {'Yes (Preview Only)' if dry_run_mode else 'No (Will Deploy)'}")

# Debug: Show raw parameter values for troubleshooting
print("\n🔍 Parameter Debug Info:")
print(f"   dry_run_mode (raw widget value): '{dbutils.widgets.get('dry_run_mode')}'")
print(f"   dry_run_mode (parsed boolean): {dry_run_mode}")
print(f"   deployment_method (raw): '{dbutils.widgets.get('deployment_method')}'")

# Ensure directories exist
print("\n📁 Ensuring directories exist...")
ensure_directory_exists(bundles_path)
print(f"   ✅ Bundles: {bundles_path}")
print("=" * 80)

In [ ]:
# ============================================================================
# Cell 2.5: Configuration Validation (Pre-flight checks)
# ============================================================================
# This runs AFTER config is loaded to validate:
# - Workspace connectivity
# - Volume paths accessible
# - Warehouse configuration valid
# - Transformed dashboards available
# ============================================================================

from helpers.config_validator import validate_and_raise

print("=" * 80)
print("CONFIGURATION VALIDATION (Pre-flight checks)")
print("=" * 80)
print()

try:
    validate_and_raise(config, verbose=True)
    print("\n✅ Configuration validation passed")
    print()
    print("📋 Next checks:")
    print("   - Cell 4.5 will verify IP whitelist status before deployment")
    print()
except Exception as e:
    print(f"\n❌ Configuration validation failed!")
    print(f"\nPlease fix the errors above and re-run this notebook.")
    raise

print("=" * 80)
print(f"   Source: {source_workspace_url}")
print(f"   Target: {target_workspace_url}")
print(f"   Volume base: {volume_base}")
print(f"   Bundle name: {bundle_name}")
print(f"   Target path: {target_parent_path}")
print(f"   Warehouse: {warehouse_id if warehouse_id else warehouse_name} {'(ID)' if warehouse_id else '(name)'}")
print(f"   Deployment method: {deployment_method.upper()}")
print(f"   Auth method: {auth_method.upper()} ({'Service Principal OAuth M2M' if auth_method == 'sp_oauth' else 'PAT Token'})")
print(f"   Dry run mode: {'Yes (Preview Only)' if dry_run_mode else 'No (Will Deploy)'}")

# Debug: Show raw parameter values for troubleshooting
print("\n🔍 Parameter Debug Info:")
print(f"   dry_run_mode (raw widget value): '{dbutils.widgets.get('dry_run_mode')}'")
print(f"   dry_run_mode (parsed boolean): {dry_run_mode}")
print(f"   deployment_method (raw): '{dbutils.widgets.get('deployment_method')}'")

# Ensure directories exist
print("\n📁 Ensuring directories exist...")
ensure_directory_exists(bundles_path)
print(f"   ✅ Bundles: {bundles_path}")
print("=" * 80)

## Cell 3a: Load Permissions (Step 1/4)

In [ ]:
import time

print("\n" + "=" * 80)
print("BUILDING UNIFIED DEPLOYMENT PACKAGES - STEP 1: LOAD PERMISSIONS")
print("=" * 80)

cell_start_time = time.time()

from helpers.deployment_package import (
    load_permissions_from_csv,
    load_schedules_from_csv
)

print("\n📋 [STEP 1/4] Loading permissions from CSV...")
print(f"   🔍 Exported path: {exported_path}")
step1_start = time.time()

try:
    permissions_map = load_permissions_from_csv(exported_path)
    step1_duration = time.time() - step1_start
    print(f"   ✅ Loaded permissions for {len(permissions_map)} dashboard(s)")
    print(f"   ⏱️  Duration: {step1_duration:.2f}s")
except FileNotFoundError as e:
    print(f"   ❌ {e}")
    print("   💡 Make sure Step 3 completed successfully with CSV generation enabled")
    raise
except Exception as e:
    print(f"   ⚠️ Warning: Could not load permissions: {e}")
    permissions_map = {}


## Cell 3b: Load Schedules (Step 2/4)

In [ ]:
print("\n" + "=" * 80)
print("STEP 2: LOAD SCHEDULES")
print("=" * 80)

print("\n📅 [STEP 2/4] Loading schedules from CSV...")
step2_start = time.time()

try:
    schedules_map = load_schedules_from_csv(exported_path)
    step2_duration = time.time() - step2_start
    print(f"   ✅ Loaded schedules for {len(schedules_map)} dashboard(s)")
    print(f"   ⏱️  Duration: {step2_duration:.2f}s")
except FileNotFoundError as e:
    step2_duration = time.time() - step2_start
    print(f"   ❌ {e}")
    print("   💡 Make sure Step 3 completed successfully with CSV generation enabled")
    raise
except Exception as e:
    step2_duration = time.time() - step2_start
    print(f"   ⚠️ Warning: Could not load schedules: {e}")
    schedules_map = {}


## Cell 3c: List Transformed Files (Step 3/4)

In [ ]:
print("\n" + "=" * 80)
print("STEP 3: LIST TRANSFORMED FILES")
print("=" * 80)

print("\n🔍 [STEP 3/4] Scanning transformed JSONs...")
print(f"   🔍 Transformed path: {transformed_path}")
step3_start = time.time()

try:
    transformed_files = dbutils.fs.ls(transformed_path)
    json_files = [f for f in transformed_files if f.name.endswith('.json') and not f.path.endswith('/')]
    step3_duration = time.time() - step3_start
    
    print(f"   ✅ Found {len(json_files)} JSON file(s)")
    print(f"   ⏱️  Duration: {step3_duration:.2f}s")
    
    if len(json_files) > 0:
        print(f"\n   📄 File list (showing first 10):")
        for i, f in enumerate(json_files[:10]):
            print(f"      {i+1}. {f.name} ({f.size} bytes)")
        if len(json_files) > 10:
            print(f"      ... and {len(json_files) - 10} more files")
except Exception as e:
    step3_duration = time.time() - step3_start
    print(f"   ❌ Could not list transformed files: {e}")
    print(f"   ⏱️  Duration: {step3_duration:.2f}s")
    import traceback
    traceback.print_exc()
    raise

print("\n✅ Step 3 complete. Run next cell for Step 4.")

## Cell 3d: Build Deployment Packages (Step 4/4)

In [ ]:
print("\n" + "=" * 80)
print("STEP 4: BUILD DEPLOYMENT PACKAGES")
print("=" * 80)

print("\n📦 [STEP 4/4] Building deployment packages...")
print(f"   🔍 Apply permissions: {apply_permissions}")
print(f"   🔍 Apply schedules: {apply_schedules}")
step4_start = time.time()

try:
    print(f"   🔄 Reading and parsing {len(json_files)} JSON files...")
    print(f"   🔄 This may take a while for large dashboards...")
    
    deployment_packages = build_deployment_packages(
        transformed_path=transformed_path,
        permissions_map=permissions_map if apply_permissions else None,
        schedules_map=schedules_map if apply_schedules else None
    )
    
    step4_duration = time.time() - step4_start
    print(f"\n   ✅ Built {len(deployment_packages)} deployment package(s)")
    print(f"   ⏱️  Duration: {step4_duration:.2f}s")
    
    # Display package details
    if deployment_packages:
        print(f"\n   📊 Package Summary:")
        package_details = []
        for pkg in deployment_packages:
            package_details.append({
                'Dashboard Name': pkg.dashboard_name,
                'Dashboard ID': pkg.dashboard_id,
                'Permissions': len(pkg.permissions),
                'Schedules': len(pkg.schedules),
                'Subscriptions': sum(len(s.subscriptions) for s in pkg.schedules)
            })
        
        df = pd.DataFrame(package_details)
        display(df)
    else:
        print("\n   ⚠️  No dashboards found in transformed path")
        print(f"   Check: {transformed_path}")
    
except Exception as e:
    step4_duration = time.time() - step4_start
    print(f"\n   ❌ Error building packages: {e}")
    print(f"   ⏱️  Duration before error: {step4_duration:.2f}s")
    import traceback
    traceback.print_exc()
    raise

# TOTAL TIMING
total_duration = time.time() - cell_start_time
print("\n" + "=" * 80)
print("⏱️  TIMING SUMMARY")
print("=" * 80)
print(f"   Step 1 (Permissions):         {step1_duration:.2f}s")
print(f"   Step 2 (Schedules):           {step2_duration:.2f}s")
print(f"   Step 3 (List Files):          {step3_duration:.2f}s")
print(f"   Step 4 (Build Packages):      {step4_duration:.2f}s")
print(f"   {'─' * 40}")
print(f"   TOTAL DURATION:               {total_duration:.2f}s")
print("=" * 80)

print("\n✅ All steps complete! Run next cell to continue with deployment.")

## Cell 4: Deployment Method Selection

In [ ]:
print("\n" + "=" * 80)
print("DEPLOYMENT METHOD SELECTION")
print("=" * 80)

print(f"\n✓ Deployment Method: {deployment_method.upper().replace('_', ' ')}")
print(f"✓ Dry Run Mode: {'Yes (Preview Only - No Resources Created)' if dry_run_mode else 'No (WILL DEPLOY RESOURCES)'}")
print(f"✓ Apply Permissions: {'Yes' if apply_permissions else 'No'}")
print(f"✓ Apply Schedules: {'Yes' if apply_schedules else 'No'}")

print(f"\nReady to deploy {len(deployment_packages)} dashboard(s)")
print(f"Target Workspace: {target_workspace_url}")
print(f"Deployment Location: {target_parent_path}")

if deployment_method == 'sdk_direct':
    print("\n📋 SDK Direct Deployment:")
    print("   • Dashboards created via API")
    print("   • Permissions applied immediately")
    print("   • Schedules created immediately")
    print("   • All in one automated flow")
elif deployment_method == 'asset_bundle':
    print("\n📋 Asset Bundle + SDK Deployment:")
    print("   • Bundle generated in UC volume")
    print("   • Dashboards deployed via SDK")
    print("   • Permissions applied via SDK")
    print("   • Schedules applied via SDK")
    print("   • Infrastructure-as-code workflow")

if dry_run_mode:
    print("\n⚠️  DRY RUN MODE ACTIVE - No resources will be created")
    print("   This is a preview only. Set dry_run_mode='false' to deploy.")

print("=" * 80)

## Cell 4.5: Pre-Deployment IP Check (Optional)

**Purpose:** Verify that the current cluster IP is whitelisted on the target workspace before attempting deployment.

**What this check does:**
- Detects the current cluster's egress IP
- Tests connectivity to target workspace
- Provides clear instructions if IP is blocked

**Skip this check:** Set `skip_ip_check: "true"` in databricks.yml (not recommended for production)

In [ ]:
# Import IP ACL manager
from helpers.ip_acl_manager import (
    check_ip_whitelist_status,
    format_status_message,
    detect_cluster_ip,
    save_cluster_ip
)

print("\n" + "="*80)
print("PRE-DEPLOYMENT IP WHITELIST CHECK")
print("="*80)

# Get skip flag from config (default: False)
skip_ip_check = config.get('deployment', {}).get('skip_ip_check', False)

# Convert string "true"/"false" to boolean if needed
if isinstance(skip_ip_check, str):
    skip_ip_check = skip_ip_check.lower() == 'true'

if skip_ip_check:
    print("\n⚠️  IP whitelist check SKIPPED (skip_ip_check=true)")
    print("   This is not recommended for production deployments.")
    print("")
else:
    try:
        print("\n🔍 Step 1: Detecting cluster IP...")
        
        # Try to detect cluster IP
        cluster_ip = detect_cluster_ip()
        
        if cluster_ip:
            print(f"   ✅ Detected cluster IP: {cluster_ip}")
            
            # Save to volume for later use
            if save_cluster_ip(cluster_ip, volume_base):
                print(f"   ✅ Saved to: {volume_base}/cluster_ip_metadata.json")
        else:
            print("   ⚠️  Could not auto-detect cluster IP")
            print("   Proceeding with connectivity test...")
        
        print("\n🔗 Step 2: Testing target workspace connectivity...")
        print(f"   Auth method: {auth_method.upper()}")
        
        # Create a test client to target workspace
        # This will fail if IP is blocked by ACL
        try:
            if auth_method == 'sp_oauth':
                # Use Service Principal OAuth M2M authentication
                from helpers.sp_oauth_auth import get_target_client_sp
                test_client = get_target_client_sp(
                    target_url=target_workspace_url,
                    secret_scope=sp_secret_scope
                )
            else:
                # Use PAT-based authentication (existing method)
                from helpers.auth import create_target_workspace_client
                test_client = create_target_workspace_client(
                    target_url=target_workspace_url,
                    secret_scope=target_workspace_secret_scope
                )
            
            # Try to connect
            user = test_client.current_user.me()
            print(f"   ✅ Successfully connected to target workspace")
            print(f"   ✅ Authenticated as: {user.user_name}")
            print(f"   ✅ IP whitelist check PASSED - deployment can proceed")
            
        except Exception as e:
            error_msg = str(e)
            
            # Check if it's an IP ACL error
            if "blocked by Databricks IP ACL" in error_msg or "Source IP address" in error_msg:
                print("\n" + "="*80)
                print("❌ IP WHITELIST CHECK FAILED")
                print("="*80)
                print("")
                if cluster_ip:
                    print(f"🔍 Your cluster IP: {cluster_ip}")
                print("")
                print("Your cluster IP is blocked by IP Access Lists on the target workspace.")
                print("")
                print("📋 REQUIRED ACTION:")
                print("")
                print("  1. Open a new terminal on your local machine")
                print("  2. Navigate to the project directory:")
                print(f"     cd \"/Users/archana.krishnamurthy/Downloads/Career & Growth/Cursor/Customer-Work/Catalog Migration\"")
                print("")
                print("  3. Run the IP whitelisting script:")
                print("     ./scripts/auto_setup_ip_acl.sh")
                print("")
                print("  4. Wait for completion (~5 minutes)")
                print("")
                print("  5. Re-run this notebook")
                print("")
                print("="*80)
                print("")
                print("💡 TIP: You can test IP detection first with:")
                print("   ./scripts/auto_setup_ip_acl.sh --dry-run")
                print("")
                
                raise Exception(
                    f"IP {cluster_ip or 'UNKNOWN'} is not whitelisted on target workspace. "
                    "Run ./scripts/auto_setup_ip_acl.sh to whitelist, then retry deployment."
                )
            else:
                # Some other connection error
                print(f"\n⚠️  Connection test failed: {error_msg}")
                print("   This may be an authentication or configuration issue.")
                print("   Proceeding with deployment - IP ACL may not be enabled.")
    
    except Exception as e:
        # Re-raise IP ACL errors
        if "not whitelisted" in str(e):
            raise
        # For other errors, warn but don't block
        print(f"\n⚠️  IP check encountered an error: {e}")
        print("   Proceeding with deployment...")

print("\n✅ Pre-deployment checks complete")
print("")

## Cell 5: SDK Direct Deployment

In [ ]:
if deployment_method == 'sdk_direct':
    print("\n" + "=" * 80)
    print(f"SDK DIRECT DEPLOYMENT {'(DRY RUN - PREVIEW ONLY)' if dry_run_mode else ''}")
    print("=" * 80)
    
    try:
        # Step 1: Authenticate to target workspace
        print("\n🔐 Step 1: Authenticating to target workspace...")
        print(f"   Auth method: {auth_method.upper()}")
        
        if auth_method == 'sp_oauth':
            # Use Service Principal OAuth M2M authentication
            from helpers.sp_oauth_auth import get_target_client_sp
            target_client = get_target_client_sp(
                target_url=target_workspace_url,
                secret_scope=sp_secret_scope
            )
        else:
            # Use PAT-based authentication (existing method)
            target_client = create_target_workspace_client(
                target_url=target_workspace_url,
                secret_scope=target_workspace_secret_scope
            )
        
        # Test connection and VERIFY we're connected to TARGET workspace
        current_user = target_client.current_user.me()
        print(f"   ✅ Authenticated as: {current_user.user_name}")
        
        # CRITICAL: Verify client is connected to TARGET workspace (not source!)
        print(f"\n🔍 CRITICAL - Workspace Connection Verification:")
        print(f"   Client host: {target_client.config.host}")
        print(f"   Expected target host: {target_workspace_url}")
        
        # Normalize URLs for comparison (remove trailing slashes)
        client_host = target_client.config.host.rstrip('/')
        expected_host = target_workspace_url.rstrip('/')
        
        if client_host == expected_host:
            print(f"   ✅ VERIFIED: Connected to TARGET workspace")
        else:
            print(f"   ❌ ERROR: Connected to WRONG workspace!")
            print(f"      Client is connected to: {client_host}")
            print(f"      Should be connected to: {expected_host}")
            raise RuntimeError(f"Client connected to wrong workspace: {client_host} (expected: {expected_host})")
        
        # DEBUG: Verify target folder is accessible
        print(f"\n🔍 DEBUG - Verifying target folder access...")
        print(f"   Target path: {target_parent_path}")
        try:
            # Try to list the parent of target path to verify access
            parent_parts = target_parent_path.rsplit('/', 1)
            parent_dir = parent_parts[0] if len(parent_parts) > 1 else '/'
            folders = target_client.workspace.list(parent_dir)
            target_folder_name = parent_parts[1] if len(parent_parts) > 1 else target_parent_path
            folder_found = any(f.path == target_parent_path for f in folders)
            print(f"   Can list parent folder ({parent_dir}): Yes")
            print(f"   Target folder exists: {folder_found}")
            if not folder_found:
                print(f"   ⚠️  WARNING: Folder not found via workspace API")
                print(f"   Will attempt to create it...")
                target_client.workspace.mkdirs(target_parent_path)
                print(f"   ✅ Created target folder: {target_parent_path}")
        except Exception as e:
            print(f"   ⚠️  Could not verify folder: {e}")
            print(f"   Attempting to create it anyway...")
            try:
                target_client.workspace.mkdirs(target_parent_path)
                print(f"   ✅ Created target folder: {target_parent_path}")
            except Exception as e2:
                print(f"   ❌ Failed to create folder: {e2}")
        
        # Step 2: Resolve warehouse ID (if not provided directly)
        print("\n🏭 Step 2: Resolving warehouse...")
        if warehouse_id:
            print(f"   ✅ Using warehouse ID from config: {warehouse_id}")
        else:
            print(f"   🔍 Resolving warehouse name: {warehouse_name}")
            warehouse_id = resolve_warehouse(target_client, warehouse_name)
            print(f"   ✅ Warehouse ID: {warehouse_id}")
        
        # Step 3: Deploy dashboards with all metadata
        print(f"\n📤 Step 3: Deploying {len(deployment_packages)} dashboard(s)...")
        
        # DEBUG: Show deployment mode status
        print(f"\n🔍 DEBUG - Deployment Mode Check:")
        print(f"   dry_run_mode variable value: {dry_run_mode}")
        print(f"   Type: {type(dry_run_mode)}")
        print(f"   Will pass to deploy_via_sdk: dry_run={dry_run_mode}")
        
        if dry_run_mode:
            print("\n   ⚠️  DRY RUN MODE - Previewing changes only\n")
        else:
            print("\n   ⚙️  LIVE MODE - Creating resources in target workspace...\n")
        
        deployment_result = deploy_via_sdk(
            client=target_client,
            packages=deployment_packages,
            target_parent_path=target_parent_path,
            warehouse_id=warehouse_id if warehouse_id else None,
            warehouse_name=warehouse_name if not warehouse_id else None,
            apply_permissions=apply_permissions,
            apply_schedules=apply_schedules,
            embed_credentials=embed_credentials,
            skip_duplicate_check=skip_duplicate_check,
            dry_run=dry_run_mode
        )
        
        # Display results
        print(f"\n{'=' * 80}")
        print(f"DEPLOYMENT {'PREVIEW' if dry_run_mode else 'RESULTS'}")
        print(f"{'=' * 80}\n")
        
        print(f"Total Dashboards: {deployment_result['total']}")
        
        if dry_run_mode:
            # In dry run, count 'dry_run' status as successful previews
            dry_run_count = len([r for r in deployment_result['results'] if r['status'] == 'dry_run'])
            error_count = len([r for r in deployment_result['results'] if r['status'] == 'error'])
            print(f"Would Create: {dry_run_count} (preview)")
            if error_count > 0:
                print(f"Validation Errors: {error_count}")
        else:
            # In live mode, show actual success/failure/skipped
            print(f"Successful: {deployment_result['successful']}")
            if deployment_result.get('skipped', 0) > 0:
                print(f"Skipped: {deployment_result['skipped']} (already exist)")
            print(f"Failed: {deployment_result['failed']}")
        
        # Show details for each dashboard
        print(f"\nDetails:\n")
        for result in deployment_result['results']:
            status_icon = '✅' if result['status'] == 'success' else '⚠️' if result['status'] in ['dry_run', 'skipped'] else '❌'
            print(f"{status_icon} {result['dashboard_name']}")
            
            if result['status'] == 'dry_run':
                print(f"   Would create dashboard")
                if 'permissions' in result['steps']:
                    print(f"   Would apply {result['steps']['permissions']} permission(s)")
                if 'schedules' in result['steps']:
                    print(f"   Would create {result['steps']['schedules']} schedule(s)")
                if 'subscriptions' in result['steps']:
                    print(f"   Would create {result['steps']['subscriptions']} subscription(s)")
            elif result['status'] == 'skipped':
                print(f"   Already exists - skipped (no changes made)")
                for error in result.get('errors', []):
                    print(f"   Note: {error}")
            elif result['status'] == 'success':
                if 'dashboard' in result['steps']:
                    print(f"   Dashboard ID: {result['steps']['dashboard']}")
                if 'permissions' in result['steps']:
                    print(f"   Applied {result['steps']['permissions']} permission(s)")
                if 'schedules' in result['steps']:
                    print(f"   Created {result['steps']['schedules']} schedule(s)")
                if 'subscriptions' in result['steps']:
                    print(f"   Created {result['steps']['subscriptions']} subscription(s)")
            else:
                for error in result.get('errors', []):
                    print(f"   Error: {error}")
            print()
        
        # Quality check: Fail job if success rate < 90% (live mode only)
        # Note: Skipped dashboards (already exist) are not counted as failures
        if not dry_run_mode and deployment_result['total'] > 0:
            created_or_updated = deployment_result['successful'] + deployment_result.get('skipped', 0)
            success_rate = (created_or_updated / deployment_result['total']) * 100
            print(f"\n📊 Success Rate: {success_rate:.1f}% (includes skipped)")
            print(f"   Successful: {deployment_result['successful']}")
            if deployment_result.get('skipped', 0) > 0:
                print(f"   Skipped: {deployment_result['skipped']} (already exist)")
            print(f"   Failed: {deployment_result['failed']}")
            
            if success_rate < 90:
                print(f"\n❌ QUALITY CHECK FAILED: Success rate ({success_rate:.1f}%) is below 90% threshold")
                print("\n🔧 Review errors above and fix issues before proceeding.")
                print("=" * 80)
                raise RuntimeError(f"Deployment quality check failed: {success_rate:.1f}% success rate (minimum: 90%)")
            else:
                print(f"   ✅ Quality check passed (threshold: 90%)")
        
        if dry_run_mode:
            print("\n✅ Dry run complete - No resources were created")
            print("   Set dry_run_mode='false' to perform actual deployment")
        else:
            print("\n✅ SDK Direct deployment complete!")
        
        print("=" * 80)
        
    except Exception as e:
        print(f"\n❌ SDK Direct deployment failed: {e}")
        import traceback
        traceback.print_exc()
        raise

else:
    print("\n⚠️  Skipping SDK Direct deployment (Asset Bundle method selected)")
    deployment_result = None

## Cell 6: Asset Bundle Generation

In [ ]:
if deployment_method == 'asset_bundle':
    print("\n" + "=" * 80)
    print("ASSET BUNDLE DEPLOYMENT PATH")
    print("=" * 80)
    
    try:
        # Step 1: Generate bundle structure
        print("\n📦 Step 1: Generating asset bundle structure...")
        
        # Extract dashboard JSONs from packages
        dashboards_for_bundle = {}
        permissions_for_bundle = {}
        
        for pkg in deployment_packages:
            dashboards_for_bundle[pkg.dashboard_id] = {
                'display_name': pkg.dashboard_name,
                'json': pkg.dashboard_json,
                'source_path': pkg.source_path
            }
            
            if apply_permissions and pkg.permissions:
                permissions_for_bundle[pkg.dashboard_id] = {
                    'dashboard_id': pkg.dashboard_id,
                    'permissions': [
                        {
                            'principal': p.principal,
                            'principal_type': p.principal_type,
                            'level': p.level
                        }
                        for p in pkg.permissions
                    ]
                }
        
        bundle_path = generate_bundle_structure(
            bundle_name=bundle_name,
            target_workspace_url=target_workspace_url,
            transformed_dashboards=dashboards_for_bundle,
            permissions_map=permissions_for_bundle,
            bundle_output_path=bundles_path,
            warehouse_id=warehouse_id,
            warehouse_name=warehouse_name,
            parent_path=target_parent_path,
            embed_credentials=embed_credentials,
            
        )
        
        print(f"   ✅ Bundle generated at: {bundle_path}")
        
        # Step 2: Validate bundle (may skip if CLI not available)
        print("\n🔍 Step 2: Validating bundle...")
        validation_result = validate_bundle(bundle_path)
        
        if validation_result.get('skipped'):
            print("   ⚠️  Bundle validation skipped (CLI not available in notebook)")
            print("   ℹ️  To validate from local machine:")
            print(f"      databricks bundle validate --profile <profile>")
        elif validation_result.get('success'):
            print("   ✅ Bundle validation passed")
        else:
            print("   ❌ Bundle validation failed")
            print(f"   Error: {validation_result.get('stderr')}")
        
        # Step 3: Deploy dashboards via SDK (bundle stays in volume)
        if not dry_run_mode:
            print("\n📤 Step 3: Deploying dashboards via SDK...")
            print(f"   Auth method: {auth_method.upper()}")
            
            # Authenticate to target using configured auth method
            if auth_method == 'sp_oauth':
                from helpers.sp_oauth_auth import get_target_client_sp
                target_client = get_target_client_sp(
                    target_url=target_workspace_url,
                    secret_scope=sp_secret_scope
                )
            else:
                target_client = create_target_workspace_client(
                    target_url=target_workspace_url,
                    secret_scope=target_workspace_secret_scope
                )
            
            # Resolve warehouse
            warehouse_id = resolve_warehouse(target_client, warehouse_name)
            
            # Deploy using SDK (without schedules - will be applied separately)
            deployment_result = deploy_via_sdk(
                client=target_client,
                packages=deployment_packages,
                target_parent_path=target_parent_path,
                warehouse_id=warehouse_id,
                apply_permissions=apply_permissions,
                apply_schedules=False,  # Apply separately
                embed_credentials=embed_credentials,
                skip_duplicate_check=skip_duplicate_check,
                dry_run=False
            )
            
            print(f"\n   ✅ Deployed {deployment_result['successful']}/{deployment_result['total']} dashboard(s)")
        else:
            print("\n⚠️  Skipping deployment (dry run mode)")
            deployment_result = None
        
        print("\n✅ Asset Bundle path complete")
        print("=" * 80)
        
    except Exception as e:
        print(f"\n❌ Asset Bundle deployment failed: {e}")
        import traceback
        traceback.print_exc()
        raise

else:
    print("\n⚠️  Skipping Asset Bundle generation (SDK Direct method selected)")
    bundle_path = None

## Cell 7: Apply Schedules & Subscriptions

In [ ]:
if deployment_method == 'asset_bundle' and apply_schedules and not dry_run_mode:
    print("\n" + "=" * 80)
    print("APPLYING SCHEDULES & SUBSCRIPTIONS")
    print("=" * 80)
    
    try:
        # Authenticate to target using configured auth method
        print("\n🔐 Authenticating to target workspace...")
        print(f"   Auth method: {auth_method.upper()}")
        
        if auth_method == 'sp_oauth':
            from helpers.sp_oauth_auth import get_target_client_sp
            target_client = get_target_client_sp(
                target_url=target_workspace_url,
                secret_scope=sp_secret_scope
            )
        else:
            target_client = create_target_workspace_client(
                target_url=target_workspace_url,
                secret_scope=target_workspace_secret_scope
            )
        
        # List deployed dashboards to get new IDs
        print("\n📋 Finding deployed dashboards...")
        deployed_dashboards = {}
        for dash in target_client.lakeview.list():
            if dash.parent_path and dash.parent_path.startswith(target_parent_path):
                deployed_dashboards[dash.display_name] = dash.dashboard_id
        
        print(f"   Found {len(deployed_dashboards)} dashboard(s) in target location")
        
        # Apply schedules for each package
        schedules_created = 0
        subscriptions_created = 0
        
        for pkg in deployment_packages:
            if pkg.schedules and pkg.dashboard_name in deployed_dashboards:
                new_dashboard_id = deployed_dashboards[pkg.dashboard_name]
                
                print(f"\n📅 Applying schedules to: {pkg.dashboard_name}")
                
                from helpers.sdk_deployer import apply_schedules_sdk
                
                scheds, subs = apply_schedules_sdk(
                    client=target_client,
                    dashboard_id=new_dashboard_id,
                    schedules=pkg.schedules
                )
                
                schedules_created += scheds
                subscriptions_created += subs
                
                print(f"   ✅ Created {scheds} schedule(s), {subs} subscription(s)")
        
        print(f"\n✅ Schedules applied successfully")
        print(f"   Total schedules: {schedules_created}")
        print(f"   Total subscriptions: {subscriptions_created}")
        print("=" * 80)
        
    except Exception as e:
        print(f"\n⚠️  Warning: Schedule application failed: {e}")
        print("   Dashboards were deployed but schedules may need manual configuration")
        import traceback
        traceback.print_exc()

elif deployment_method == 'sdk_direct':
    print("\n⚠️  Schedules already applied during SDK Direct deployment")
elif dry_run_mode:
    print("\n⚠️  Skipping schedule application (dry run mode)")
else:
    print("\n⚠️  Schedule application skipped (apply_schedules=false)")

## Cell 8: Verify Deployment & Report

In [ ]:
# ============================================================================
# IMPORTANT: Verification is SKIPPED in dry run mode because:
# 1. No resources were actually created (dry run only previews)
# 2. Verification would find 0 dashboards (misleading)
# 3. API calls to target workspace are unnecessary for preview
# 4. Dry run should be fast and non-invasive
# ============================================================================

if not dry_run_mode:
    print("\n" + "=" * 80)
    print("DEPLOYMENT VERIFICATION & SUMMARY")
    print("=" * 80)
    
    try:
        # Connect to target workspace using configured auth method
        print("\n🔐 Connecting to target workspace...")
        print(f"   Auth method: {auth_method.upper()}")
        
        if auth_method == 'sp_oauth':
            from helpers.sp_oauth_auth import get_target_client_sp
            target_client = get_target_client_sp(
                target_url=target_workspace_url,
                secret_scope=sp_secret_scope
            )
        else:
            target_client = create_target_workspace_client(
                target_url=target_workspace_url,
                secret_scope=target_workspace_secret_scope
            )
        
        # Verify deployed dashboards using deployment results (FAST - no scanning)
        print("\n🔍 Verifying deployed dashboards...")
        print(f"   Using deployment results (fast method - no workspace scan)")
        
        verified_dashboards = []
        
        if deployment_result and 'results' in deployment_result:
            for result in deployment_result['results']:
                if result.get('status') == 'success' and 'dashboard' in result.get('steps', {}):
                    dashboard_id = result['steps']['dashboard']
                    
                    # Verify this specific dashboard exists (direct API call, not list scan)
                    try:
                        dash = target_client.lakeview.get(dashboard_id)
                        verified_dashboards.append({
                            'Name': dash.display_name,
                            'ID': dash.dashboard_id,
                            'Path': dash.parent_path,
                            'Status': 'Published' if hasattr(dash, 'lifecycle_state') else 'Active'
                        })
                    except Exception as e:
                        print(f"      ⚠️  Could not verify dashboard {dashboard_id}: {e}")
        
        if verified_dashboards:
            print(f"\n✅ Verified {len(verified_dashboards)} dashboard(s)\n")
            df = pd.DataFrame(verified_dashboards)
            display(df)
        else:
            print("\n⚠️  No dashboards verified")
            print(f"   Check deployment results above for dashboard IDs")
            print(f"   Expected location: {target_parent_path}")
        
        # Generate comprehensive summary report
        print("\n" + "=" * 80)
        print("MIGRATION SUMMARY REPORT")
        print("=" * 80)
        
        print(f"\n📊 Deployment Details:")
        print(f"   Method: {deployment_method.upper().replace('_', ' ')}")
        print(f"   Source: {source_workspace_url}")
        print(f"   Target: {target_workspace_url}")
        print(f"   Location: {target_parent_path}")
        
        print(f"\n📈 Migration Statistics:")
        if deployment_result:
            print(f"   Dashboards Created: {deployment_result['successful']}/{deployment_result['total']}")
            if deployment_result.get('skipped', 0) > 0:
                print(f"   Dashboards Skipped: {deployment_result['skipped']} (already exist - not updated)")
            print(f"   Dashboards Verified: {len(verified_dashboards)}")
        else:
            print(f"   Dashboards: {len(verified_dashboards)}/{len(deployment_packages)}")
        
        if deployment_result and deployment_method == 'sdk_direct':
            total_perms = sum(
                r['steps'].get('permissions', 0) 
                for r in deployment_result['results'] 
                if r.get('status') == 'success'
            )
            total_scheds = sum(
                r['steps'].get('schedules', 0) 
                for r in deployment_result['results'] 
                if r.get('status') == 'success'
            )
            total_subs = sum(
                r['steps'].get('subscriptions', 0) 
                for r in deployment_result['results'] 
                if r.get('status') == 'success'
            )
            
            print(f"   Permissions Applied: {total_perms}")
            print(f"   Schedules Created: {total_scheds}")
            print(f"   Subscriptions Created: {total_subs}")
        
        print("\n✅ Migration Complete!")
        
        print("\n📋 Next Steps:")
        print("   1. Open target workspace UI")
        print(f"   2. Navigate to {target_parent_path}")
        print("   3. Open each dashboard to verify:")
        print("      • Dashboard loads without errors")
        print("      • Visualizations render correctly")
        print("      • Data queries execute successfully")
        print("      • Permissions are correct (click Share button)")
        print("      • Schedules are configured (click Schedule button)")
        print("   4. Test data with new catalog/schema references")
        print("   5. Monitor scheduled refreshes")
        
        print("\n" + "=" * 80)
        
    except Exception as e:
        print(f"\n⚠️  Verification failed: {e}")
        print("   Deployment may have succeeded but verification encountered an error")
        import traceback
        traceback.print_exc()

else:
    # ========================================================================
    # DRY RUN MODE: Verification is intentionally SKIPPED
    # ========================================================================
    print("\n" + "=" * 80)
    print("DRY RUN SUMMARY (VERIFICATION SKIPPED)")
    print("=" * 80)
    
    print("\n✅ Dry run completed successfully\n")
    
    print("⚠️  IMPORTANT: Verification was NOT performed because:")
    print("   • No resources were created in the target workspace (preview only)")
    print("   • Verification would list 0 dashboards (misleading)")
    print("   • Dry run mode should be fast and non-invasive")
    print("   • API calls to target workspace are unnecessary for preview")
    
    if deployment_result:
        print("\n📊 Preview Results:")
        print(f"   Would create: {deployment_result['total']} dashboard(s)")
        
        total_perms = sum(len(p.permissions) for p in deployment_packages)
        total_scheds = sum(len(p.schedules) for p in deployment_packages)
        total_subs = sum(sum(len(s.subscriptions) for s in p.schedules) for p in deployment_packages)
        
        print(f"   Would apply: {total_perms} permission(s)")
        print(f"   Would create: {total_scheds} schedule(s)")
        print(f"   Would create: {total_subs} subscription(s)")
        
        # Show any errors that occurred during dry run
        failed = [r for r in deployment_result['results'] if r['status'] == 'error']
        if failed:
            print(f"\n⚠️  {len(failed)} dashboard(s) had validation errors:")
            for r in failed[:5]:  # Show first 5
                print(f"      • {r['dashboard_name']}: {r['errors'][0] if r['errors'] else 'Unknown error'}")
    
    print("\n📋 To perform actual deployment:")
    print("   databricks bundle run generate_deploy -t dev --var=\"dry_run_mode=false\" --profile source-workspace")
    
    print("\n" + "=" * 80)

---

## ✅ Notebook Complete

### What This Notebook Does

1. **Loads Configuration**: From databricks.yml or interactive widgets
2. **Builds Deployment Packages**: Unified data structures with dashboards + permissions + schedules
3. **Authenticates**: Cross-workspace authentication with fallback chain
4. **Deploys**: Two paths available:
   - **SDK Direct**: All resources via API in one flow
   - **Asset Bundle**: Bundle + SDK for IaC workflows
5. **Verifies**: Checks deployed dashboards and generates report

### Execution Modes

**CLI Mode (Automated):**
```bash
# Dry run (preview - default, safe)
databricks bundle run generate_deploy -t dev --profile source-workspace

# Live deployment (explicit flag required)
databricks bundle run generate_deploy -t dev --var="dry_run_mode=false" --profile source-workspace

# Asset Bundle method + Live
databricks bundle run generate_deploy -t dev --var="deployment_method=asset_bundle" --var="dry_run_mode=false" --profile source-workspace
```

**Interactive Mode (Manual):**
1. Open this notebook in Databricks UI
2. Set widget values at top (deployment_method, dry_run_mode, etc.)
3. Run all cells

### Testing

See `TESTING_GUIDE.md` for complete testing scenarios.

---